# Morning Reports & TV Dashboard Generator
This notebook integrates the daily report generation from `Local.ipynb` and the initial TV dashboard rendering of `update_dashboard.py` into a single, unified workflow. All overlapping data queries (NBP prices, forward curves, weather forecasts, table inputs) are performed exactly once to reduce API load and eliminate redundancy.

### Output Artifacts:
1. **OneDrive report images** (`SandWNBP.png`, `forward_nbp.png`, `table.png`, `wandt.png`)
2. **Shared drive combined image** (`I:\Trading\FlogasNews\Daily_Image\Combined_IMG_cropped.png`)
3. **TV Display HTML** (`index.html`)
4. **Dashboard Cache** (`dashboard_cache.json`) for the background updater service.

In [1]:
import os
import re
import sys
import time
import json
import csv
import datetime
from datetime import timedelta
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
from io import StringIO
from jinja2 import Environment, FileSystemLoader

# Try importing proprietary and news APIs
try:
    import icepython as icexl
except ImportError:
    icexl = None
    print("Warning: icepython library not found!")

try:
    from eventregistry import EventRegistry, QueryArticles, RequestArticlesInfo
except ImportError:
    EventRegistry = None
    print("Warning: eventregistry library not found!")

# Set Paths
WORKSPACE_DIR = os.getcwd()
CSV_PATH = os.path.join(WORKSPACE_DIR, 'commodity_prices.csv')
CACHE_PATH = os.path.join(WORKSPACE_DIR, 'dashboard_cache.json')
TEMPLATE_NAME = 'tv_template.html'
OUTPUT_NAME = 'index.html'

ONEDRIVE_DIR = r"C:\Users\diarmuid.egan\OneDrive - Flogas Ireland\Microsoft Teams Chat Files\Pictures\Daily_tv"
SHARED_IMG_PATH = r"I:\Trading\FlogasNews\Daily_Image\Combined_IMG.png"
print("Setup complete. Workspace:", WORKSPACE_DIR)

Setup complete. Workspace: C:\Users\diarmuid.egan\Music\scripts\TV


## 1. Fetch & Plot NBP Daily Prices (Summer 27 / Winter 26)

In [2]:
print("Fetching NBP Summer/Winter Daily Prices...")
current_date = datetime.date.today()
ed = current_date.strftime('%Y-%m-%d')
sd = (current_date - timedelta(days=365)).strftime('%Y-%m-%d')

fields = ['volume', 'close']
my_data = icexl.get_timeseries(['GWMS 1!-ICE'], fields, granularity='D', start_date=sd, end_date=ed)
my_data2 = icexl.get_timeseries(['GWMS 2!-ICE'], fields, granularity='D', start_date=sd, end_date=ed)

dates, vols, values = zip(*my_data[1:])  
dates2, vols2, values2 = zip(*my_data2[1:])  
formatted_dates = [date for date in dates]

num_points = len(formatted_dates)
indices = [0, num_points//3, 2*num_points//3, num_points-1]
selected_dates = [formatted_dates[i] for i in indices]

nbp_daily_fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,  
    vertical_spacing=0.1,  
    row_heights=[0.99, 0.01] 
)
nbp_daily_fig.add_trace(go.Scatter(x=formatted_dates, y=values, mode='lines+markers', marker=dict(color='deepskyblue'), name='Winter 26'), row=1, col=1)
nbp_daily_fig.add_trace(go.Scatter(x=formatted_dates, y=values2, mode='lines+markers', marker=dict(color='darkviolet'), name='Summer 27'), row=1, col=1)

nbp_daily_fig.update_layout(
    title={
        'text': 'Winter 26 and Summer 27 Daily NBP Prices 12 Month Lag',
        'x': 0.5,
        'xanchor': 'center'
    },
    legend=dict(
        x=0.5,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.8)', 
        bordercolor='lightgray',
        borderwidth=1
    ),
    yaxis_title='p/therm',
    showlegend=True,
    xaxis=dict(
        tickmode='array',
        tickvals=[formatted_dates[i] for i in indices],
        ticktext=selected_dates
    ),
    width=1600, 
    height=1600,
    font=dict(size=32)
)
print("NBP Daily Timeseries loaded.")

Fetching NBP Summer/Winter Daily Prices...


NBP Daily Timeseries loaded.


## 2. Fetch & Plot NBP Forward Curve WoW

In [3]:
print("Fetching NBP Forward Curve WoW...")
sd_7 = (current_date - timedelta(days=7)).strftime('%Y-%m-%d')
lom = ['GWM ' + str(i) + '!-ICE' for i in range(1, 13)]

my_data_wow = icexl.get_timeseries(lom, ['Last'], granularity='D', start_date=sd_7, end_date=sd_7)
last_week_p = my_data_wow[1][1:]

wow_pull = icexl.get_quotes(lom, ['ICE Theoretical Price'])
current_theo_p = [i[1] for i in wow_pull[1:]]

def get_next_12_months():
    start_date = datetime.date.today() + datetime.timedelta(days=1)
    months_list = []
    curr_d = start_date
    for _ in range(12):
        curr_m = curr_d.month
        curr_y = curr_d.year
        next_m = curr_m + 1
        next_y = curr_y
        if next_m > 12:
            next_m = 1
            next_y += 1
        curr_d = datetime.date(next_y, next_m, 1)
        months_list.append(curr_d.strftime('%b'))
    return months_list

labels = get_next_12_months()

nbp_forward_fig = go.Figure()
nbp_forward_fig.add_trace(go.Scatter(
    x=labels, 
    y=last_week_p, 
    mode='lines+markers', 
    name='Last Week',
    marker=dict(color='deepskyblue'),
    line=dict(color='deepskyblue')
))
nbp_forward_fig.add_trace(go.Scatter(
    x=labels, 
    y=current_theo_p, 
    mode='lines+markers', 
    name='Current Price',
    marker=dict(color='darkviolet'),
    line=dict(color='darkviolet')
))

nbp_forward_fig.update_layout(
    title={
        'text': 'Forward NBP Curve Change Week on Week',
        'x': 0.5,
        'xanchor': 'center'
    },
    xaxis_title='Month',
    yaxis_title='P/th',
    legend=dict(
        x=0.5,
        y=0.98,
        xanchor='center',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.8)', 
        bordercolor='lightgray',
        borderwidth=1,
        orientation='h'
    ),
    hovermode='x unified',
    width=1600, 
    height=800,
    font=dict(size=32)
)
print("NBP Forward Curve loaded.")

Fetching NBP Forward Curve WoW...


NBP Forward Curve loaded.


## 3. Fetch & Plot Wind & Temperature Forecast

In [4]:
print("Fetching Wind & Temperature Forecast...")
enappsys_urls = {
    'Temp': "https://app.enappsys.com/datadownload?code=isem/weather/tempair/national/history&currency=EUR&delimiter=comma&minavmax=false&pass=211225245243225231229237225238177178183161&res=hh&tag=csv&timezone=WET&user=diarmuid.egan@flogas.ie", 
    'Wind': "https://app.enappsys.com/datadownload?code=isem/elec/renewables/wind/onshore/forecast/history&currency=EUR&delimiter=comma&minavmax=false&pass=211225245243225231229237225238177178183161&res=hh&tag=csv&timezone=WET&user=diarmuid.egan@flogas.ie"
}

def gen_enappsys_url(base):
    curr_date = datetime.date.today()
    start_date = curr_date.strftime('%Y%m%d0000')
    end_date = (curr_date + timedelta(days=6)).strftime('%Y%m%d2330')
    return f"{enappsys_urls[base]}&start={start_date}&end={end_date}"

def get_enappsys_data(base):
    response = requests.get(gen_enappsys_url(base))
    if response.status_code != 200:
        raise Exception(f"Failed to fetch {base} forecast: {response.status_code}")
    df = pd.read_csv(StringIO(response.text))
    df['Date (WET)'] = df['Date (WET)'].str.strip('[]')
    df['Date (WET)'] = pd.to_datetime(df['Date (WET)'], format='%d/%m/%Y %H:%M')
    df = df.drop(index=0).reset_index(drop=True)
    if base == 'Temp':
        df['LATEST_TEMP'] = df['LATEST'].astype(float)
        df['Date'] = df['Date (WET)']
        return df[['Date','LATEST_TEMP']]
    elif base == 'Wind':
        df['LATEST_WIND'] = df['LATEST FORECAST (EnAppSys)'].astype(float)
        df['Date'] = df['Date (WET)']
        return df[['Date','LATEST_WIND']]

Wind_df = get_enappsys_data('Wind')
Temp_df = get_enappsys_data('Temp')

temp_summary = Temp_df.groupby(Temp_df['Date'].dt.date).agg(
    min_temp=('LATEST_TEMP', 'min'),
    max_temp=('LATEST_TEMP', 'max')
).reset_index()
temp_summary['Date'] = pd.to_datetime(temp_summary['Date']) + pd.Timedelta(hours=12)

wind_temp_fig = make_subplots(specs=[[{"secondary_y": True}]])
wind_temp_fig.add_trace(go.Scatter(
    x=Wind_df['Date'],
    y=Wind_df['LATEST_WIND'],
    name='Wind',
    mode='lines',
    line=dict(color='darkviolet')
), secondary_y=False)

wind_temp_fig.add_trace(go.Bar(
    x=temp_summary['Date'],
    y=temp_summary['max_temp'] - temp_summary['min_temp'],
    base=temp_summary['min_temp'],
    name='Temp Range',
    marker=dict(color='deepskyblue', opacity=0.6),
    width=1000 * 3600 * 4
), secondary_y=True)

for i, row in temp_summary.iterrows():
    wind_temp_fig.add_annotation(x=row['Date'], y=row['max_temp'], yref="y2", text=f"{row['max_temp']:.1f}", showarrow=False, font=dict(color="black"), yshift=10)
    wind_temp_fig.add_annotation(x=row['Date'], y=row['min_temp'], yref="y2", text=f"{row['min_temp']:.1f}", showarrow=False, font=dict(color="black"), yshift=-10)

wind_temp_fig.update_layout(
    title={'text': 'Wind and Temperature Forecast', 'x': 0.5, 'xanchor': 'center'}, 
    showlegend=False, 
    width=1600, 
    height=800,
    yaxis=dict(showgrid=False),  
    yaxis2=dict(showgrid=False),
    font=dict(size=32)
)
wind_temp_fig.update_yaxes(title_text="Wind (MW)", secondary_y=False, range=[0, 5000])
wind_temp_fig.update_yaxes(title_text="Temperature (°C)", secondary_y=True)
print("Wind & Temperature Forecast loaded.")

Fetching Wind & Temperature Forecast...


Wind & Temperature Forecast loaded.


## 4. Fetch & Plot Baseload Table

In [5]:
print("Fetching Baseload Table Data...")
ss_months = {
    'apr': 20.51, 'may': 16.17, 'jun': 16.41, 'jul': 16.54, 
    'aug': 16.33, 'sep': 16.32, 'oct': 16.31, 'nov': 16.31, 
    'dec': 16.83, 'jan': 17.55, 'feb': 18.33, 'mar': 19.00 
}
Margin = 4.5


def get_next_24_months():
    start_date = datetime.date.today().replace(day=1)  
    months_list = []
    for i in range(1, 25):
        year = start_date.year + (start_date.month + i - 1) // 12
        month = (start_date.month + i - 1) % 12 + 1
        date = datetime.date(year, month, 1)
        months_list.append(date.strftime('%b-%y'))
    return months_list

table_labels = get_next_24_months()
ss = [ss_months.get(label[:3].lower(), 16.0) for label in table_labels]

def Baseload_gen(gas, carbon, spark, fx):
    t1 = gas / (2.93071 * 0.4913 * fx)
    t2 = (0.18404 / 0.4913) * carbon
    return float(t1) + float(t2) + float(Margin) + float(spark)

margin = np.linspace(1.9, 3.4, 24)
lom24 = ['GWM ' + str(i) + '!-ICE' for i in range(1, 25)]
fpf = ['ICE Theoretical Price']
f24_pull = icexl.get_quotes(lom24, fpf)
Gas_forward_theo_p = [i[1] for i in f24_pull[1:]]

CarbonM1_pull = icexl.get_quotes(['ECF 1!-ICN'], fpf)
CarbonDec27_pull = icexl.get_quotes(['ECF 13!-ICN'], fpf)
Carbon = np.linspace(float(CarbonM1_pull[1][1]), float(CarbonDec27_pull[1][1]), 24)

GBPEUR = float(icexl.get_quotes(['EURGBP@FXP A0-FX'], ['Last'])[1][1])
Gas = [round(float(Gas_forward_theo_p[i] + margin[i]), 1) for i in range(24)]
Baseload = [round(float(Baseload_gen(Gas[i], Carbon[i], ss[i], GBPEUR)), 1) for i in range(24)]

columns = [['<b>NPB Gas p/th</b>', '<b>Baseload €/MWh</b>']]
for i in range(len(table_labels)):
    columns.append([Gas[i], Baseload[i]])

baseload_table_fig = go.Figure(data=[go.Table( 
    header=dict(
        values=['<b></b>'] + [f'<b>{label}</b>' for label in table_labels],
        fill_color='rgba(173, 216, 230, 0.5)',
        line_color='black',
        line_width=2,
        align='center', 
        font=dict(size=32, color='black', family='Arial')
    ),
    cells=dict(
        values=columns,
        align=['center', 'center'], 
        fill_color='rgba(226, 233, 243, 0.3)',
        line_color='white',
        line_width=2,
        font=dict(size=32, color='black', family='Arial'),
        height=20)
)])
baseload_table_fig.update_layout(
    width=3200,
    height=600,
    margin=dict(l=20, r=20, t=20, b=20)
)
print("Baseload table data loaded.")

Fetching Baseload Table Data...


Baseload table data loaded.


## 5. Fetch Forward Diesel WoW (TV Dashboard Box 3)

In [6]:
print("Fetching Forward Diesel/Heating Oil Curve WoW...")
sd_gas = (current_date - timedelta(days=7)).strftime('%Y-%m-%d')
lom_gas = ['GAS ' + str(i) + '!-ICE' for i in range(1, 13)]

my_data_gas = icexl.get_timeseries(lom_gas, ['Last'], granularity='D', start_date=sd_gas, end_date=sd_gas)
last_week_gas_p = my_data_gas[1][1:]

wow_pull_gas = icexl.get_quotes(lom_gas, ['ICE Theoretical Price'])
current_theo_gas_p = [i[1] for i in wow_pull_gas[1:]]

tv_diesel_fig = go.Figure()
tv_diesel_fig.add_trace(go.Scatter(
    x=labels, 
    y=last_week_gas_p, 
    mode='lines+markers', 
    name='Last Week',
    marker=dict(color='deepskyblue'),
    line=dict(color='deepskyblue')
))
tv_diesel_fig.add_trace(go.Scatter(
    x=labels, 
    y=current_theo_gas_p, 
    mode='lines+markers', 
    name='Current Price',
    marker=dict(color='darkviolet'),
    line=dict(color='darkviolet')
))

tv_diesel_fig.update_layout(
    title={'text': 'Forward Diesel/Heating Oil Curve Change Week on Week', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Month', yaxis_title='USD ($) /MT',
    legend=dict(x=0.5, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    hovermode='x unified',
    font=dict(size=24), autosize=True, margin=dict(l=50, r=30, t=60, b=95)
)
print("Diesel Curve loaded.")

Fetching Forward Diesel/Heating Oil Curve WoW...


Diesel Curve loaded.


## 6. Fetch News Headlines (TV Ticker)

In [7]:
print("Fetching EventRegistry news headlines...")
news_headlines = []
if EventRegistry is not None:
    try:
        er = EventRegistry(apiKey='1312ee84-a9f9-4fc6-95a4-0ed13b135ec3')
        nat_gas_concept_uri = er.getConceptUri("Natural Gas")
        q = QueryArticles(conceptUri=nat_gas_concept_uri, lang="eng")
        q.setRequestedResult(RequestArticlesInfo(sortBy="date", count=100))
        results = er.execQuery(q)
        
        unique_list = []
        seen_titles = set()
        if results.get('articles') and results['articles'].get('results'):
            for article in results['articles']['results']:
                title = article['title'].strip()
                source_name = article['source']['title']
                if re.search('gas', title, re.IGNORECASE) and title not in seen_titles:
                    unique_list.append(f"{title} - {source_name}")
                    seen_titles.add(title)
        news_headlines = unique_list + unique_list
    except Exception as e:
        print(f"Error fetching headlines: {e}")

if not news_headlines:
    news_headlines = ["No gas news headlines found today."]
print(f"Retrieved {len(news_headlines)//2 if EventRegistry else 0} unique news articles.")

Fetching EventRegistry news headlines...


Retrieved 11 unique news articles.


## 7. Scrape Brent & TTF Gas Prices (TV Dashboard Box 4)

In [8]:
print("Scraping TradingEconomics real-time commodity prices...")
brent_val, ttf_val = None, None
try:
    url = "https://tradingeconomics.com/commodities"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    }
    r = requests.get(url, headers=headers, timeout=10)
    if r.status_code == 200:
        soup = BeautifulSoup(r.text, 'html.parser')
        brent_row = soup.find('tr', {'data-symbol': 'CO1:COM'})
        if brent_row:
            brent_val = float(brent_row.find('td', {'id': 'p'}).get_text(strip=True).replace(',', ''))
        
        ttf_row = soup.find('tr', {'data-symbol': 'NGEU:COM'})
        if ttf_row:
            ttf_val = float(ttf_row.find('td', {'id': 'p'}).get_text(strip=True).replace(',', ''))
except Exception as e:
    print(f"Error scraping commodities: {e}")

if brent_val is not None and ttf_val is not None:
    now_ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    file_exists = os.path.exists(CSV_PATH)
    with open(CSV_PATH, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['Timestamp', 'Brent', 'TTF_Gas'])
        writer.writerow([now_ts, brent_val, ttf_val])
    print(f"Scrape success: Brent = {brent_val} | TTF Gas = {ttf_val}")
else:
    print("Scrape failed or returned empty. Using existing CSV records for TV plot.")

Scraping TradingEconomics real-time commodity prices...


Scrape success: Brent = 72.662 | TTF Gas = 40.83


## 8. Export Report Images (OneDrive & Shared Drive Combined Layout)

In [9]:
print("--- Saving Report Images ---")

# 1. Save individual graphs to OneDrive
if not os.path.exists(ONEDRIVE_DIR):
    try:
        os.makedirs(ONEDRIVE_DIR)
    except:
        pass

try:
    nbp_daily_fig.write_image(os.path.join(ONEDRIVE_DIR, "SandWNBP.png"))
    nbp_forward_fig.write_image(os.path.join(ONEDRIVE_DIR, "forward_nbp.png"))
    baseload_table_fig.write_image(os.path.join(ONEDRIVE_DIR, "table.png"))
    wind_temp_fig.write_image(os.path.join(ONEDRIVE_DIR, "wandt.png"))
    print("Successfully saved NBP, Table, and Weather graphs to OneDrive.")
except Exception as e:
    print(f"Warning: Could not write OneDrive images: {e}")

# 2. Build 3x2 Combined Image for Shared Drive
try:
    specs = [
        [{"rowspan": 2, "type": "xy"}, {"type": "xy"}],
        [None, {"type": "xy", "secondary_y": True}],
        [{"colspan": 2, "type": "table"}, None]
    ]

    combined_fig = make_subplots(
        rows=3, cols=2,
        specs=specs,
        subplot_titles=("Summer 26 and Winter 26 NBP", " NPB Forward Curve WoW p/th", None, None),
        vertical_spacing=0.06,
        horizontal_spacing=0.05
    )

    # Copy traces
    for trace in nbp_daily_fig.data:
        trace_copy = trace
        trace_copy.showlegend = True
        trace_copy.legend = "legend"
        trace_copy.legendgroup = "group1"
        combined_fig.add_trace(trace_copy, row=1, col=1)

    for trace in nbp_forward_fig.data:
        trace_copy = trace
        trace_copy.showlegend = True
        trace_copy.legend = "legend2"
        trace_copy.legendgroup = "group2"
        combined_fig.add_trace(trace_copy, row=1, col=2)

    combined_fig.add_trace(go.Scatter(
        x=Wind_df['Date'],
        y=Wind_df['LATEST_WIND'],
        name='Wind',
        mode='lines',
        line=dict(color='darkviolet'),
        showlegend=False
    ), row=2, col=2, secondary_y=False)

    combined_fig.add_trace(go.Bar(
        x=temp_summary['Date'],
        y=temp_summary['max_temp'] - temp_summary['min_temp'],
        base=temp_summary['min_temp'],
        name='Temp Range',
        marker=dict(color='deepskyblue', opacity=0.6),
        width=4 * 3600 * 1000,
        showlegend=False
    ), row=2, col=2, secondary_y=True)

    for _, row_ in temp_summary.iterrows():
        combined_fig.add_annotation(
            x=row_['Date'], 
            y=row_['max_temp'],
            xref='x3',
            yref='y4',
            text=f"{row_['max_temp']:.1f}",
            showarrow=False,
            font=dict(color="black", size=32),
            yshift=10
        )
        combined_fig.add_annotation(
            x=row_['Date'],
            y=row_['min_temp'],
            xref='x3',
            yref='y4',
            text=f"{row_['min_temp']:.1f}",
            showarrow=False,
            font=dict(color="black", size=32),
            yshift=-10
        )

    for t in baseload_table_fig.data:
        combined_fig.add_trace(t, row=3, col=1)

    combined_fig.update_xaxes(title_text="", row=2, col=2)
    combined_fig.update_yaxes(title_text="Wind Gen (MW)", row=2, col=2, secondary_y=False, range=[0, 5000], title_font=dict(size=36), tickfont=dict(size=32))
    combined_fig.update_yaxes(title_text="Temperature (°C)", row=2, col=2, secondary_y=True, title_font=dict(size=36), tickfont=dict(size=32))

    for i, annotation in enumerate(combined_fig.layout.annotations):
        if i == 0:
            annotation.x = 0.223
            annotation.xanchor = 'center'
            annotation.font.size = 36
            annotation.font.color = 'black'
        elif i == 1:
            annotation.x = 0.73
            annotation.xanchor = 'center'
            annotation.font.size = 36
            annotation.font.color = 'black'

    combined_fig.update_layout(
        height=2500,
        width=6400,
        legend=dict(
            x=0.34,
            y=0.96,
            xanchor='center',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='black',
            borderwidth=1, font=dict(size=32)
        ),
        legend2=dict(
            x=0.73,
            y=0.96,
            xanchor='center',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.8)',
            bordercolor='black',
            borderwidth=1, font=dict(size=32)
        )
    )

    combined_fig.update_xaxes(title_font=dict(size=32), tickfont=dict(size=32))
    combined_fig.update_yaxes(title_font=dict(size=32), tickfont=dict(size=32))

    shared_dir = os.path.dirname(SHARED_IMG_PATH)
    if not os.path.exists(shared_dir):
        try:
            os.makedirs(shared_dir)
        except:
            pass

    combined_fig.write_image(SHARED_IMG_PATH)
    print(f"Successfully saved Combined_IMG to {SHARED_IMG_PATH}")

    # Crop whitespace and save
    out_cropped_path = os.path.splitext(SHARED_IMG_PATH)[0] + "_cropped.png"
    img = Image.open(SHARED_IMG_PATH)
    w, h = img.size 
    box = (0, 0, max(0, w - 290), max(0, h - 450))
    cropped = img.crop(box)
    cropped.save(out_cropped_path)
    print(f"Successfully saved cropped combined image to {out_cropped_path}")

except Exception as e:
    print(f"Error writing Combined Shared Drive image: {e}")

--- Saving Report Images ---


Successfully saved NBP, Table, and Weather graphs to OneDrive.


Successfully saved Combined_IMG to I:\Trading\FlogasNews\Daily_Image\Combined_IMG.png


Successfully saved cropped combined image to I:\Trading\FlogasNews\Daily_Image\Combined_IMG_cropped.png


## 9. Compile TV Dashboard HTML & Save Cache

In [10]:
# Generate HTML fragments
# Reset layout sizes for responsive TV display and set proper margins/fonts/legends/X-axes
nbp_daily_fig.update_layout(width=None, height=None, autosize=True, font=dict(size=24), margin=dict(l=60, r=40, t=60, b=55), legend=dict(x=0.5, y=0.98, xanchor='center', yanchor='top', orientation='h', bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1))
nbp_daily_fig.update_xaxes(showticklabels=True, title=None, tickfont=dict(size=14))

nbp_forward_fig.update_layout(width=None, height=None, autosize=True, font=dict(size=24), margin=dict(l=60, r=40, t=60, b=55), legend=dict(x=0.5, y=0.98, xanchor='center', yanchor='top', orientation='h', bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1))
nbp_forward_fig.update_xaxes(showticklabels=True, title=None, tickfont=dict(size=14))

tv_diesel_fig.update_layout(width=None, height=None, autosize=True, font=dict(size=24), margin=dict(l=60, r=40, t=60, b=55), legend=dict(x=0.5, y=0.98, xanchor='center', yanchor='top', orientation='h', bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1))
tv_diesel_fig.update_xaxes(showticklabels=True, title=None, tickfont=dict(size=14))

fig1_html = nbp_daily_fig.to_html(full_html=False, include_plotlyjs='cdn')
fig2_html = nbp_forward_fig.to_html(full_html=False, include_plotlyjs=False)
fig3_html = tv_diesel_fig.to_html(full_html=False, include_plotlyjs=False)

try:
    from update_dashboard import generate_commodity_chart
    fig4 = generate_commodity_chart()
    fig4.update_layout(width=None, height=None, autosize=True, font=dict(size=24), margin=dict(l=80, r=80, t=60, b=55), legend=dict(x=0.5, y=0.98, xanchor='center', yanchor='top', orientation='h', bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1, font=dict(size=16)))
    fig4.update_xaxes(showticklabels=True, title=None, tickfont=dict(size=14))
    fig4_html = fig4.to_html(full_html=False, include_plotlyjs=False)
except Exception as e:
    print(f"Error generating commodity chart: {e}")
    fig4_html = "<div>Error generating commodity chart. Check console logs.</div>"

# Write cached blocks for background update service
cache_data = {
    'fig1_html': fig1_html,
    'fig2_html': fig2_html,
    'fig3_html': fig3_html,
    'headlines': news_headlines
}
try:
    with open(CACHE_PATH, 'w', encoding='utf-8') as f:
        json.dump(cache_data, f, ensure_ascii=False, indent=2)
    print(f"Cached files saved to {CACHE_PATH} for background loop service.")
except Exception as e:
    print(f"Error saving cache: {e}")

# Render the index.html page
try:
    env = Environment(loader=FileSystemLoader(WORKSPACE_DIR))
    template = env.get_template(TEMPLATE_NAME)
    html_content = template.render(
        fig1_html=fig1_html,
        fig2_html=fig2_html,
        fig3_html=fig3_html,
        fig4_html=fig4_html,
        headlines=news_headlines
    )
    output_file_path = os.path.join(WORKSPACE_DIR, OUTPUT_NAME)
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    print(f"Successfully generated Flogas Energy Dashboard: {output_file_path}")
    # Push files to GitHub
    import github_sync
    github_sync.push_to_github(WORKSPACE_DIR)
except Exception as e:
    print(f"Critical error rendering dashboard: {e}")


Generating Brent & TTF Gas Prices real-time chart...
Cached files saved to C:\Users\diarmuid.egan\Music\scripts\TV\dashboard_cache.json for background loop service.
Successfully generated Flogas Energy Dashboard: C:\Users\diarmuid.egan\Music\scripts\TV\index.html
[2026-06-25 10:55:36] Pushing updated files to GitHub...


Successfully pushed to GitHub.
